In [1]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold


def generate_kaggle_submission():

    # ============================================================
    # 1. Load Data
    # ============================================================
    base_dir = "/kaggle/input/competitions/march-machine-learning-mania-2026"
    if not os.path.exists(base_dir):
        base_dir = "./"

    m_reg = pd.read_csv(os.path.join(base_dir, "MRegularSeasonDetailedResults.csv"))
    w_reg = pd.read_csv(os.path.join(base_dir, "WRegularSeasonDetailedResults.csv"))

    m_tourney = pd.read_csv(os.path.join(base_dir, "MNCAATourneyCompactResults.csv"))
    w_tourney = pd.read_csv(os.path.join(base_dir, "WNCAATourneyCompactResults.csv"))

    m_seeds = pd.read_csv(os.path.join(base_dir, "MNCAATourneySeeds.csv"))
    w_seeds = pd.read_csv(os.path.join(base_dir, "WNCAATourneySeeds.csv"))

    m_tourney = m_tourney[m_tourney["Season"] >= 2003].copy()
    w_tourney = w_tourney[w_tourney["Season"] >= 2010].copy()

    # ============================================================
    # 2. Seed Processing
    # ============================================================
    def parse_seed(seed):
        return int(seed[1:3])

    for df in [m_seeds, w_seeds]:
        df["SeedNum"] = df["Seed"].apply(parse_seed)

    # ============================================================
    # 3. Regular Season Aggregates
    # ============================================================
    def extract_team_metrics(df):

        stat_cols = [
            "Score", "FGM", "FGA", "FGM3", "FGA3",
            "FTM", "FTA", "OR", "DR", "Ast",
            "TO", "Stl", "Blk", "PF"
        ]

        win_df = df[["Season", "WTeamID"] + ["W" + c for c in stat_cols]].copy()
        win_df.columns = ["Season", "TeamID"] + stat_cols
        win_df["wins"] = 1
        win_df["losses"] = 0

        loss_df = df[["Season", "LTeamID"] + ["L" + c for c in stat_cols]].copy()
        loss_df.columns = ["Season", "TeamID"] + stat_cols
        loss_df["wins"] = 0
        loss_df["losses"] = 1

        combined = pd.concat([win_df, loss_df], ignore_index=True)

        grouped = combined.groupby(["Season", "TeamID"], as_index=False).sum()
        grouped["games"] = grouped["wins"] + grouped["losses"]
        grouped["win_rate"] = grouped["wins"] / grouped["games"]

        for c in stat_cols:
            grouped[f"avg_{c}"] = grouped[c] / grouped["games"]

        grouped["fg_pct"] = grouped["FGM"] / grouped["FGA"].replace(0, 1)
        grouped["fg3_pct"] = grouped["FGM3"] / grouped["FGA3"].replace(0, 1)
        grouped["ft_pct"] = grouped["FTM"] / grouped["FTA"].replace(0, 1)
        grouped["ast_to_ratio"] = grouped["Ast"] / grouped["TO"].replace(0, 1)

        keep_cols = (
            ["Season", "TeamID", "win_rate"]
            + [f"avg_{c}" for c in stat_cols]
            + ["fg_pct", "fg3_pct", "ft_pct", "ast_to_ratio"]
        )

        return grouped[keep_cols]

    m_metrics = extract_team_metrics(m_reg)
    w_metrics = extract_team_metrics(w_reg)

    # ============================================================
    # 4. Feature Builders
    # ============================================================
    def add_matchup_features(df, metrics_df, seeds_df):

        df = df.merge(
            seeds_df[["Season", "TeamID", "SeedNum"]],
            left_on=["Season", "Team1"],
            right_on=["Season", "TeamID"],
            how="left"
        ).rename(columns={"SeedNum": "T1_Seed"}).drop("TeamID", axis=1)

        df = df.merge(
            seeds_df[["Season", "TeamID", "SeedNum"]],
            left_on=["Season", "Team2"],
            right_on=["Season", "TeamID"],
            how="left"
        ).rename(columns={"SeedNum": "T2_Seed"}).drop("TeamID", axis=1)

        df[["T1_Seed", "T2_Seed"]] = df[["T1_Seed", "T2_Seed"]].fillna(16)
        df["Seed_diff"] = df["T1_Seed"] - df["T2_Seed"]

        df = df.merge(
            metrics_df,
            left_on=["Season", "Team1"],
            right_on=["Season", "TeamID"],
            how="left"
        ).drop("TeamID", axis=1)

        df = df.merge(
            metrics_df,
            left_on=["Season", "Team2"],
            right_on=["Season", "TeamID"],
            how="left",
            suffixes=("_T1", "_T2")
        ).drop("TeamID", axis=1)

        metric_cols = [c for c in df.columns if c.endswith("_T1")]
        for c in metric_cols:
            base = c.replace("_T1", "")
            df[f"{base}_diff"] = df[c] - df[f"{base}_T2"]

        return df

    def build_train_features(tourney_df, metrics_df, seeds_df):

        wins = tourney_df[["Season", "WTeamID", "LTeamID"]].copy()
        wins.columns = ["Season", "Team1", "Team2"]
        wins["target"] = 1

        losses = tourney_df[["Season", "LTeamID", "WTeamID"]].copy()
        losses.columns = ["Season", "Team1", "Team2"]
        losses["target"] = 0

        df = pd.concat([wins, losses], ignore_index=True)

        return add_matchup_features(df, metrics_df, seeds_df)

    def build_test_features(df, metrics_df, seeds_df):
        df = df.copy()
        return add_matchup_features(df, metrics_df, seeds_df)

    # Build training
    m_train = build_train_features(m_tourney, m_metrics, m_seeds)
    w_train = build_train_features(w_tourney, w_metrics, w_seeds)

    train_df = pd.concat([m_train, w_train], ignore_index=True)

    features = [c for c in train_df.columns
                if c not in ["Season", "Team1", "Team2", "target"]]

    # ============================================================
    # 5. Train Model
    # ============================================================
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    models = []

    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "learning_rate": 0.03,
        "num_leaves": 31,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "verbosity": -1
    }

    for tr_idx, val_idx in skf.split(train_df[features], train_df["target"]):

        tr_data = lgb.Dataset(train_df.iloc[tr_idx][features],
                              label=train_df.iloc[tr_idx]["target"])

        val_data = lgb.Dataset(train_df.iloc[val_idx][features],
                               label=train_df.iloc[val_idx]["target"])

        model = lgb.train(
            params,
            tr_data,
            num_boost_round=2000,
            valid_sets=[val_data],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )

        models.append(model)

    # ============================================================
    # 6. Submission
    # ============================================================
    sub = pd.read_csv(os.path.join(base_dir, "SampleSubmissionStage1.csv"))

    sub["Season"] = sub["ID"].str.split("_").str[0].astype(int)
    sub["Team1"] = sub["ID"].str.split("_").str[1].astype(int)
    sub["Team2"] = sub["ID"].str.split("_").str[2].astype(int)

    submission_ids = sub["ID"].copy()

    m_sub = build_test_features(sub[sub["Team1"] < 3000].copy(),
                                m_metrics, m_seeds)

    w_sub = build_test_features(sub[sub["Team1"] >= 3000].copy(),
                                w_metrics, w_seeds)

    test_df = pd.concat([m_sub, w_sub], ignore_index=True)

    test_features = test_df[features].fillna(0)

    preds = np.mean([m.predict(test_features) for m in models], axis=0)
    preds = np.clip(preds, 0.02, 0.98)

    submission = pd.DataFrame({
        "ID": submission_ids,
        "Pred": preds
    })

    submission.to_csv("submission.csv", index=False)
    print("Submission saved.")


if __name__ == "__main__":
    generate_kaggle_submission()

Submission saved.
